# Handwriting Recognition (TrOCR)

Notebook-first refactor for handwritten text recognition using a real Kaggle dataset download, strict data verification, TrOCR inference, and honest CER/WER evaluation.

## Dataset Source

Kaggle dataset: https://www.kaggle.com/datasets/landenpatterson/handwritten-text-recognition

This notebook downloads data in-code and validates image files, labels, and train/val/test splits before inference and evaluation.

In [ ]:
import importlib
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    package_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

ensure_package('kagglehub')
ensure_package('pandas')
ensure_package('numpy')
ensure_package('matplotlib')
ensure_package('PIL', 'Pillow')
ensure_package('torch')
ensure_package('transformers')
ensure_package('evaluate')
print('Dependencies are ready.')

In [ ]:
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PROJECT_DIR = Path.cwd()
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Project directory: {PROJECT_DIR}')

## Real Dataset Download From Kaggle

In [ ]:
import kagglehub

if not os.getenv('KAGGLE_USERNAME') or not os.getenv('KAGGLE_KEY'):
    raise EnvironmentError('Missing Kaggle credentials. Set KAGGLE_USERNAME and KAGGLE_KEY.')

dataset_root = Path(kagglehub.dataset_download('landenpatterson/handwritten-text-recognition'))
print(f'Dataset root: {dataset_root}')

## Data Validation: Images, Labels, and Splits

In [ ]:
image_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tif', '.tiff'}
image_paths = [p for p in dataset_root.rglob('*') if p.suffix.lower() in image_extensions]
if len(image_paths) == 0:
    raise RuntimeError('No image files found in downloaded dataset.')

label_files = [p for p in dataset_root.rglob('*') if p.suffix.lower() in {'.csv', '.tsv', '.txt'}]
if len(label_files) == 0:
    raise RuntimeError('No candidate label files (.csv/.tsv/.txt) found in dataset.')

corrupt_count = 0
sample_n = min(300, len(image_paths))
for p in random.sample(image_paths, sample_n):
    try:
        with Image.open(p) as img:
            img.verify()
    except Exception:
        corrupt_count += 1
if corrupt_count > 0:
    raise RuntimeError(f'Corrupted images found during sample verification: {corrupt_count}')

print(f'Total images discovered: {len(image_paths)}')
print(f'Candidate label files: {len(label_files)}')

In [ ]:
def safe_read_table(path_obj):
    ext = path_obj.suffix.lower()
    if ext == '.csv':
        return pd.read_csv(path_obj)
    if ext == '.tsv':
        return pd.read_csv(path_obj, sep='\t')
    if ext == '.txt':
        try:
            return pd.read_csv(path_obj, sep='\t')
        except Exception:
            return pd.read_csv(path_obj)
    return pd.DataFrame()

def find_col(df, candidates):
    low_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in low_map:
            return low_map[c]
    return None

pairs = []
for lf in label_files:
    try:
        tmp = safe_read_table(lf)
    except Exception:
        continue
    if tmp.empty:
        continue

    image_col = find_col(tmp, ['image', 'image_name', 'filename', 'file', 'img', 'image_path'])
    text_col = find_col(tmp, ['label', 'text', 'transcription', 'identity', 'word', 'gt', 'ground_truth'])
    if image_col is None or text_col is None:
        continue

    part = tmp[[image_col, text_col]].copy()
    part.columns = ['image_name', 'label']
    part['source_file'] = str(lf)
    pairs.append(part)

if len(pairs) == 0:
    raise RuntimeError('Could not parse any usable label table with image and text columns.')

labels_df = pd.concat(pairs, ignore_index=True)
labels_df['image_name'] = labels_df['image_name'].astype(str).str.strip()
labels_df['label'] = labels_df['label'].astype(str).str.strip()
labels_df = labels_df[labels_df['label'] != ''].copy()
labels_df = labels_df.drop_duplicates(subset=['image_name', 'label']).reset_index(drop=True)

def resolve_image(name):
    direct = dataset_root / name
    if direct.exists():
        return str(direct)
    base = Path(name).name
    matches = list(dataset_root.rglob(base))
    if len(matches) > 0:
        return str(matches[0])
    return None

labels_df['image_path'] = labels_df['image_name'].apply(resolve_image)
labels_df = labels_df.dropna(subset=['image_path']).reset_index(drop=True)
if len(labels_df) == 0:
    raise RuntimeError('No labels could be matched to real image files.')

def infer_split(path_str):
    x = path_str.lower()
    if 'train' in x:
        return 'train'
    if 'val' in x or 'valid' in x or 'dev' in x:
        return 'val'
    if 'test' in x:
        return 'test'
    return 'unknown'

labels_df['split'] = labels_df['image_path'].apply(infer_split)

if (labels_df['split'] == 'unknown').all():
    shuf = labels_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    n = len(shuf)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)
    shuf.loc[:n_train - 1, 'split'] = 'train'
    shuf.loc[n_train:n_train + n_val - 1, 'split'] = 'val'
    shuf.loc[n_train + n_val:, 'split'] = 'test'
    labels_df = shuf

split_counts = labels_df['split'].value_counts().to_dict()
if split_counts.get('train', 0) == 0 or split_counts.get('val', 0) == 0 or split_counts.get('test', 0) == 0:
    raise RuntimeError(f'Split verification failed: {split_counts}')

print(f'Usable labeled samples: {len(labels_df)}')
print(f'Split counts: {split_counts}')
display(labels_df.head(8))

In [ ]:
train_df = labels_df[labels_df['split'] == 'train'].copy().reset_index(drop=True)
val_df = labels_df[labels_df['split'] == 'val'].copy().reset_index(drop=True)
test_df = labels_df[labels_df['split'] == 'test'].copy().reset_index(drop=True)

viz_df = train_df.sample(n=min(9, len(train_df)), random_state=SEED)
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for ax, (_, row) in zip(axes.flatten(), viz_df.iterrows()):
    img = Image.open(row['image_path']).convert('RGB')
    ax.imshow(img)
    ax.set_title(str(row['label'])[:40])
    ax.axis('off')
for ax in axes.flatten()[len(viz_df):]:
    ax.axis('off')
plt.tight_layout()
samples_png = ARTIFACTS_DIR / 'sample_grid.png'
plt.savefig(samples_png, dpi=140)
plt.show()

## TrOCR Inference Setup

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-handwritten')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten').to(DEVICE)
model.eval()
print('TrOCR model loaded.')

In [ ]:
import evaluate

cer_metric = evaluate.load('cer')
wer_metric = evaluate.load('wer')

def predict_text(image_path):
    image = Image.open(image_path).convert('RGB')
    pixel_values = processor(images=image, return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)
    pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return pred.strip()

## Real Evaluation (CER/WER + Qualitative Review)

In [ ]:
eval_df = val_df.sample(n=min(200, len(val_df)), random_state=SEED).copy().reset_index(drop=True)
predictions = []
for image_path in eval_df['image_path'].tolist():
    predictions.append(predict_text(image_path))

references = eval_df['label'].astype(str).tolist()
eval_df['prediction'] = predictions

cer_value = float(cer_metric.compute(predictions=predictions, references=references))
wer_value = float(wer_metric.compute(predictions=predictions, references=references))
exact_match = float((eval_df['prediction'].str.strip() == eval_df['label'].str.strip()).mean())

print(f'Validation samples evaluated: {len(eval_df)}')
print(f'CER: {cer_value:.4f}')
print(f'WER: {wer_value:.4f}')
print(f'Exact match rate: {exact_match:.4f}')
display(eval_df[['image_path', 'label', 'prediction']].head(12))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
qual_df = eval_df.sample(n=min(6, len(eval_df)), random_state=SEED)
for ax, (_, row) in zip(axes.flatten(), qual_df.iterrows()):
    img = Image.open(row['image_path']).convert('RGB')
    ax.imshow(img)
    ax.set_title(f"GT: {str(row['label'])[:28]}\nPR: {str(row['prediction'])[:28]}")
    ax.axis('off')
for ax in axes.flatten()[len(qual_df):]:
    ax.axis('off')
plt.tight_layout()
qual_png = ARTIFACTS_DIR / 'qualitative_predictions.png'
plt.savefig(qual_png, dpi=140)
plt.show()

## Save Real Outputs

In [ ]:
pred_csv = ARTIFACTS_DIR / 'val_predictions.csv'
eval_df.to_csv(pred_csv, index=False)

metrics = {
    'dataset': 'landenpatterson/handwritten-text-recognition',
    'num_images_found': int(len(image_paths)),
    'num_labeled_samples': int(len(labels_df)),
    'split_counts': {k: int(v) for k, v in split_counts.items()},
    'num_eval_samples': int(len(eval_df)),
    'cer': cer_value,
    'wer': wer_value,
    'exact_match_rate': exact_match
}

metrics_json = ARTIFACTS_DIR / 'metrics.json'
with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

manifest = {
    'dataset_url': 'https://www.kaggle.com/datasets/landenpatterson/handwritten-text-recognition',
    'dataset_root': str(dataset_root),
    'prediction_csv': str(pred_csv),
    'metrics_json': str(metrics_json),
    'sample_grid_png': str(samples_png),
    'qualitative_png': str(qual_png)
}

manifest_json = ARTIFACTS_DIR / 'artifact_manifest.json'
with open(manifest_json, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Saved outputs:')
print(f'- {pred_csv}')
print(f'- {metrics_json}')
print(f'- {manifest_json}')
print(f'- {samples_png}')
print(f'- {qual_png}')

## Limitations and Next Improvements

- This notebook runs direct pretrained TrOCR inference; fine-tuning can improve domain-specific handwriting quality.
- Label schema can vary by Kaggle version; parsing is robust but may need minor column mapping updates for uncommon exports.
- For production, add batching, confidence filtering, and latency profiling for real-time deployment.